<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🛑 动态中断：`interrupt()`

> **`interrupt_before` = "到这里统一停车。"**
>
> **`interrupt()` = "node 根据现场情况，主动踩刹车，还能拿到人的回复。"**

</div>

### 一、静态 vs 动态

| | `interrupt_before=["tools"]` | `interrupt()` |
|---|---|---|
| 谁决定停？ | graph（compile 时） | node（runtime） |
| 看不看数据？ | 不看，每次到都停 | 看 state，按条件停 |
| 能带信息给人？ | ❌ | ✅ payload |
| 能拿到人的回复？ | ❌ | ✅ 返回值 |
| 续跑 | `invoke(None, cfg)` | `invoke(Command(resume=值), cfg)` |

```text
interrupt_before：$1 → 停 | $10 → 停 | $10000 → 停（全停）
interrupt()     ：$1 → 过 | $10 → 过 | $10000 → 停（条件停）
```

### 二、写法：节点内调用 `interrupt(payload)`

In [ ]:
from langgraph.types import interrupt, Command

def payment_node(state):
    if state["amount"] > 10000:
        approved = interrupt({                     # 暂停；payload 给人看
            "reason": "Large payment requires approval",
            "amount": state["amount"],
        })
        if not approved:                           # 人 resume 的值就是 approved
            return {"status": "rejected"}
    return {"status": "paid"}

```text
payment_node → 金额 > 1 万 → interrupt(payload) → 暂停 → 人审批 → Command(resume=True/False) → 继续
```

| 场景 | 条件 | 暂停原因 |
|---|---|---|
| 🚀 部署生产 | `env == "prod"` | 确认部署吗？ |
| 📧 发邮件 | `recipient_count > 1000` | 确认群发吗？ |
| 💰 花钱 | `cost > $500` | 确认购买吗？ |
| 🗑️ 删文件 | `delete_count > 100` | 确认删除吗？ |

本质：**conditional human-in-the-loop**。

### 三、暂停时发生了什么（`thread_id="1"`）

1. 保存 checkpoint
2. payload 记入 `result["__interrupt__"]`，也可 `graph.get_state(cfg).tasks[0].interrupts` 读
3. `get_state(cfg).next == ("payment_node",)`，thread 保持 alive，可以以后再续跑

**必须有 checkpointer + thread_id**，否则无法恢复：thread 1 paused ⏸️ · thread 2 running ▶️ · thread 3 finished ✅。

### 四、续跑

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

class State(TypedDict):
    amount: int
    status: str

builder = StateGraph(State)
builder.add_node("payment_node", payment_node)
builder.add_edge(START, "payment_node")
builder.add_edge("payment_node", END)
graph = builder.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "1"}}
result = graph.invoke({"amount": 20000}, cfg)
result["__interrupt__"][0].value                  # {'reason': 'Large payment requires approval', 'amount': 20000}

graph.invoke(Command(resume=True), cfg)           # → {'amount': 20000, 'status': 'paid'}

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**续跑时节点从头重跑**：`interrupt()` 之前的代码会再执行一次（副作用要幂等或放到 `interrupt()` 之后）；这一次 `interrupt()` 直接返回 resume 的值，不再暂停。

</div>

### 五、与 Super Step 的关系

```text
A → B, C (并行) → D
```
B 里调用 `interrupt()`：本 super step 暂停，checkpoint 保存，`next == ('B',)`。实测续跑时 **只有 B 重跑，C 不会重跑**（C 的结果已保存），之后才进 D。
不是"小异常"，它影响整个 graph 的调度。

### 六、实用场景：Bug-Fix Agent 🐛

In [ ]:
def deploy_node(state):
    if state["branch"] == "main":
        approved = interrupt("Deploying to main branch - requires approval")
        if not approved:
            return {"status": "cancelled"}
    # ... 继续部署

Slack 通知 → 你点 Approve（`Command(resume=True)`）→ 继续部署。典型的 production 用法 ✅

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：`interrupt_before` = 到点统一停（调试用）；**`interrupt(payload)` = node 按条件主动停，`Command(resume=值)` 续跑，值就是 `interrupt()` 的返回值**；依赖 checkpointer + thread_id；续跑时节点从头重跑。

</div>